# 03 — Model Training & Comparison

Train four model families and compare on the held-out **test set**.

| Model | Role |
|---|---|
| Logistic Regression | Linear baseline; coefficient interpretability |
| Random Forest | Non-linear benchmark; reliable feature importance |
| XGBoost | Primary candidate; gradient boosting |
| LightGBM | Speed-optimised alternative; native categoricals |

**Class imbalance strategy:** `class_weight='balanced'` / `scale_pos_weight=9.08` (ratio 900773:99227)

In [ ]:
import numpy as np
import pandas as pd
import json, joblib, os, time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, accuracy_score,
                              roc_curve, precision_recall_curve,
                              ConfusionMatrixDisplay, classification_report)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
SEED = 42

SCALE_POS_WEIGHT = 900773 / 99227   # ~9.08  for XGBoost / LightGBM

In [ ]:
X_train = np.load('artifacts/X_train.npy')
X_val   = np.load('artifacts/X_val.npy')
X_test  = np.load('artifacts/X_test.npy')
y_train = np.load('artifacts/y_train.npy')
y_val   = np.load('artifacts/y_val.npy')
y_test  = np.load('artifacts/y_test.npy')

with open('artifacts/proc_cols.json') as f:
    PROC_COLS = json.load(f)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')
print(f'Churn rates  train={y_train.mean():.3%}  val={y_val.mean():.3%}  test={y_test.mean():.3%}')

## 1. Evaluation Helper

All models are evaluated with the same function. **Primary metrics:** ROC-AUC and PR-AUC. Threshold is tuned on the validation set to maximise F1.

In [ ]:
def evaluate(model, X_val, y_val, X_test, y_test, model_name):
    # Probabilities
    prob_val  = model.predict_proba(X_val)[:, 1]
    prob_test = model.predict_proba(X_test)[:, 1]

    # Tune threshold on validation set (maximise F1 for churn class)
    thresholds = np.linspace(0.1, 0.9, 81)
    f1s = [f1_score(y_val, (prob_val >= t).astype(int), pos_label=1, zero_division=0)
           for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]

    y_pred_test = (prob_test >= best_t).astype(int)

    metrics = {
        'model':       model_name,
        'threshold':   round(best_t, 3),
        'roc_auc':     round(roc_auc_score(y_test, prob_test), 4),
        'pr_auc':      round(average_precision_score(y_test, prob_test), 4),
        'f1_churn':    round(f1_score(y_test, y_pred_test, pos_label=1), 4),
        'recall_churn':round(recall_score(y_test, y_pred_test, pos_label=1), 4),
        'prec_churn':  round(precision_score(y_test, y_pred_test, pos_label=1, zero_division=0), 4),
        'accuracy':    round(accuracy_score(y_test, y_pred_test), 4),
    }
    return metrics, prob_test, y_pred_test

results = []
probs   = {}   # store for ROC / PR curve plots

## 2. Model 1 — Logistic Regression (Baseline)

In [ ]:
t0 = time.time()
lr = LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                        max_iter=1000, solver='lbfgs', random_state=SEED, n_jobs=-1)
lr.fit(X_train, y_train)
elapsed = round(time.time() - t0, 1)
print(f'Logistic Regression trained in {elapsed}s')

m, prob, pred = evaluate(lr, X_val, y_val, X_test, y_test, 'LogisticRegression')
m['train_time_s'] = elapsed
results.append(m)
probs['LogisticRegression'] = prob

print(pd.Series(m).to_string())
print()
print(classification_report(y_test, pred, target_names=['Not Churned','Churned']))

## 3. Model 2 — Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, max_depth=12,
                             min_samples_leaf=50, class_weight='balanced',
                             n_jobs=-1, random_state=SEED)
rf.fit(X_train, y_train)
elapsed = round(time.time() - t0, 1)
print(f'Random Forest trained in {elapsed}s')

m, prob, pred = evaluate(rf, X_val, y_val, X_test, y_test, 'RandomForest')
m['train_time_s'] = elapsed
results.append(m)
probs['RandomForest'] = prob

print(pd.Series(m).to_string())
print()
print(classification_report(y_test, pred, target_names=['Not Churned','Churned']))

## 4. Model 3 — XGBoost

In [ ]:
t0 = time.time()
xgb = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric='aucpr', early_stopping_rounds=20,
    random_state=SEED, n_jobs=-1, verbosity=0
)
xgb.fit(X_train, y_train,
        eval_set=[(X_val, y_val)], verbose=False)
elapsed = round(time.time() - t0, 1)
print(f'XGBoost trained in {elapsed}s  |  best iteration: {xgb.best_iteration}')

m, prob, pred = evaluate(xgb, X_val, y_val, X_test, y_test, 'XGBoost')
m['train_time_s'] = elapsed
results.append(m)
probs['XGBoost'] = prob

print(pd.Series(m).to_string())
print()
print(classification_report(y_test, pred, target_names=['Not Churned','Churned']))

## 5. Model 4 — LightGBM

In [ ]:
t0 = time.time()
lgbm = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
    is_unbalance=True, random_state=SEED, n_jobs=-1, verbosity=-1
)
lgbm.fit(X_train, y_train,
         eval_set=[(X_val, y_val)],
         callbacks=[])
elapsed = round(time.time() - t0, 1)
print(f'LightGBM trained in {elapsed}s')

m, prob, pred = evaluate(lgbm, X_val, y_val, X_test, y_test, 'LightGBM')
m['train_time_s'] = elapsed
results.append(m)
probs['LightGBM'] = prob

print(pd.Series(m).to_string())
print()
print(classification_report(y_test, pred, target_names=['Not Churned','Churned']))

## 6. Comparison Table

In [ ]:
compare_df = pd.DataFrame(results).set_index('model')
display_cols = ['roc_auc','pr_auc','f1_churn','recall_churn','prec_churn','accuracy','threshold','train_time_s']
print('=== Model Comparison (test set) ===')
print(compare_df[display_cols].to_string())

# Highlight winner
winner = compare_df['roc_auc'].idxmax()
print(f'\n>>> Best ROC-AUC: {winner} ({compare_df.loc[winner,"roc_auc"]})')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Test Set', fontweight='bold', fontsize=13)
for ax, metric, color in zip(axes,
        ['roc_auc', 'pr_auc', 'f1_churn'],
        ['steelblue', 'teal', 'darkorange']):
    compare_df[metric].sort_values().plot(kind='barh', ax=ax, color=color,
                                          edgecolor='white', alpha=0.85)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10.colors

# ROC curves
for i, (name, prob) in enumerate(probs.items()):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=colors[i], linewidth=2)
axes[0].plot([0,1],[0,1],'k--',linewidth=0.8)
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend()

# Precision-Recall curves
for i, (name, prob) in enumerate(probs.items()):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.4f})', color=colors[i], linewidth=2)
axes[1].axhline(y_test.mean(), color='black', linestyle='--', linewidth=0.8, label='Baseline (random)')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Feature importance from RF and XGBoost
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, name in [(axes[0], rf, 'Random Forest'), (axes[1], xgb, 'XGBoost')]:
    imp = pd.Series(model.feature_importances_, index=PROC_COLS).sort_values(ascending=True)
    imp.tail(15).plot(kind='barh', ax=ax, color='tomato', edgecolor='white', alpha=0.85)
    ax.set_title(f'{name} — Top 15 Feature Importances', fontweight='bold')
    ax.set_xlabel('Importance')

plt.tight_layout(); plt.show()

In [ ]:
os.makedirs('artifacts', exist_ok=True)
joblib.dump(lr,   'artifacts/model_lr.pkl')
joblib.dump(rf,   'artifacts/model_rf.pkl')
joblib.dump(xgb,  'artifacts/model_xgb.pkl')
joblib.dump(lgbm, 'artifacts/model_lgbm.pkl')

compare_df.to_csv('artifacts/model_comparison.csv')
print('Models saved to artifacts/')
print('Comparison table saved to artifacts/model_comparison.csv')
print(f'\nBest model for tuning: {winner}')